In [1]:
import pandas as pd

In [2]:
s1 = pd.read_csv("../data/sample/sample_source1.tsv", sep='\t')
s2 = pd.read_csv("../data/sample/sample_source2.tsv", sep='\t')
s3 = pd.read_csv("../data/sample/sample_source3.tsv", sep='\t')
gt = pd.read_csv("../data/sample/sample_ground_truth.tsv", sep='\t')


In [9]:
gt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   source1_entity_id   10000 non-null  object
 1   matched_entity_ids  10000 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB


In [3]:
import sys
sys.path.append("..")

from src.normalization import normalize_pipeline

In [4]:
s1_normalized = normalize_pipeline(s1)
s2_normalized = normalize_pipeline(s2)
s3_normalized = normalize_pipeline(s3)

In [5]:
s1_normalized.head()

,entity_id,business_name,business_address,country,clean_name,clean_address,geo_block_key,phonetic_hash,search_document
0,S1-602779913,Indore Sago Pvt Ltd,"Indore, Madhya Pradesh, Office No. 411-A, Indo...",India,indore sago ltd,indore madhya pradesh office no 411 a indore 4...,india_unknown,INTR SK LTT,indore sago ltd indore madhya pradesh office n...
1,S1-405144457,Services Satyanarayan Utility,"Rajiv Colony, Hari Om Forging, Ballabgarh, Far...",India,services satyanarayan utility,rajiv colony hari om forging ballabgarh farida...,india_haryana,SRFSS STYNRYN UTLT,services satyanarayan utility rajiv colony har...
2,S1-242670339,Abad Partnership Inc,"Flagstaff, 4870 Indian Drive, AZ",US,abad partnership inc,flagstaff 4870 indian drive az,us_az,ABT PRTNRXP INK,abad partnership inc flagstaff 4870 indian dri...
3,S1-886582923,XVF Impex Private Limited,"51/46, 6Th Cross, Bhavani Mansion, Wilson Gard...",India,xvf impex ltd,51 46 6th cross bhavani mansion wilson garden ...,india_karnataka,SFF IMPKS LTT,xvf impex ltd 51 46 6th cross bhavani mansion ...
4,S1-367561647,Aja (India) Silk Pvt Ltd,"Iv/217, Andipillikavu Moothakunnam P.O., North...",India,aja india silk ltd,iv 217 andipillikavu moothakunnam p o north pa...,india_kerala,AJ INT SLK LTT,aja india silk ltd iv 217 andipillikavu mootha...


In [ ]:
s1_normalized['geo_block_key'].value_counts()

In [ ]:
from src.blocking import run_multi_index_generation

In [ ]:
# Execute Index 1 and Index 2
candidate_pairs = run_multi_index_generation(s1_normalized, s2_normalized, top_k=15)

# Inspect the result
print(candidate_pairs.sort_values(by='bm25_text_score', ascending=False).head(10))

## Candidate pool using all 3 sources

In [ ]:
import pandas as pd

def generate_global_candidate_pool(s1, s2, s3, top_k=15):
    """
    Executes the global BM25 indices across all source combinations 
    and standardizes the output into a single candidate pool.
    """
    print("="*40)
    print("STAGE 1: S1 vs S2")
    print("="*40)
    pairs_12 = run_multi_index_generation(s1, s2, top_k)
    
    print("\n" + "="*40)
    print("STAGE 2: S1 vs S3")
    print("="*40)
    pairs_13 = run_multi_index_generation(s1, s3, top_k)
    
    print("\n" + "="*40)
    print("STAGE 3: S2 vs S3")
    print("="*40)
    pairs_23 = run_multi_index_generation(s2, s3, top_k)
    
    # Standardize column names so they stack perfectly
    standard_columns = ['entity_A', 'entity_B', 'bm25_text_score', 'bm25_phonetic_score']
    
    if not pairs_12.empty: pairs_12.columns = standard_columns
    if not pairs_13.empty: pairs_13.columns = standard_columns
    if not pairs_23.empty: pairs_23.columns = standard_columns
    
    # Combine into a single master candidate pool
    print("\nMerging all cross-source candidate pairs...")
    master_pool = pd.concat([pairs_12, pairs_13, pairs_23], ignore_index=True)
    
    # Drop exact duplicates if multiple combinations caught the same pair
    master_pool = master_pool.drop_duplicates(subset=['entity_A', 'entity_B'])
    
    print(f"Total Unique Candidate Pairs Generated: {len(master_pool)}")
    return master_pool

# Execute the full pool generation
global_candidates_df = generate_global_candidate_pool(s1_normalized, s2_normalized, s3_normalized, top_k=15)

# Inspect the top matches
print(global_candidates_df.sort_values(by='bm25_text_score', ascending=False).head(10))

## Validating the candidate pairs obtained

In [ ]:


def validate_blocking_recall(candidates_df, ground_truth_df, gt_col_1='id1', gt_col_2='id2'):
    """
    Calculates what percentage of the ground truth matches were successfully 
    retrieved by our multi-index search engines.
    """
    # 1. Extract the true positive pairs from ground truth
    # We sort them alphabetically so (A, B) is treated the same as (B, A)
    print("Standardizing ground truth pairs...")
    true_pairs = set()
    for _, row in ground_truth_df.iterrows():
        pair = tuple(sorted([str(row[gt_col_1]), str(row[gt_col_2])]))
        true_pairs.add(pair)
        
    total_true = len(true_pairs)
    print(f"Total True Matches in Ground Truth: {total_true}")
    
    # 2. Extract the candidate pairs generated by our pipeline
    print("Standardizing generated candidate pairs...")
    candidate_pairs = set()
    for _, row in candidates_df.iterrows():
        pair = tuple(sorted([str(row['entity_A']), str(row['entity_B'])]))
        candidate_pairs.add(pair)
        
    total_candidates = len(candidate_pairs)
    print(f"Total Candidate Pairs Generated: {total_candidates}")
    
    # 3. Calculate Recall (Intersection)
    captured_pairs = true_pairs.intersection(candidate_pairs)
    captured_count = len(captured_pairs)
    
    recall = (captured_count / total_true) * 100 if total_true > 0 else 0
    
    print("\n" + "="*30)
    print("BLOCKING PIPELINE METRICS")
    print("="*30)
    print(f"-> True Matches Captured: {captured_count} / {total_true}")
    print(f"-> RECALL: {recall:.2f}%")
    
    # Calculate Reduction Ratio (How much we shrank the trillion-pair search space)
    # Assuming sample size of roughly ~10,000 per dataset for this run
    theoretical_max = 10000 * 10000 * 3 # Rough estimate for S1xS2, S1xS3, S2xS3
    reduction_ratio = 100 - ((total_candidates / theoretical_max) * 100)
    print(f"-> Search Space Reduction: ~{reduction_ratio:.4f}%")
    
    return captured_pairs, true_pairs - captured_pairs

# Execute validation (Change 'id1' and 'id2' to match your actual ground truth column names)
# Check g.columns if you aren't sure!
captured, missed = validate_blocking_recall(global_candidates_df, gt, gt_col_1='source1_entity_id', gt_col_2='matched_entity_ids')

In [ ]:
gt.head()

In [ ]:
# 1. Check how many missed pairs actually exist in our current subset
s1_ids = set(s1_normalized['entity_id'])
s2_ids = set(s2_normalized['entity_id'])
s3_ids = set(s3_normalized['entity_id'])
all_available_ids = s1_ids | s2_ids | s3_ids

verifiable_misses = []
for pair in missed:
    # A pair is only "verifiable" if BOTH IDs are actually in the dataframes we processed
    if pair[0] in all_available_ids and pair[1] in all_available_ids:
        verifiable_misses.append(pair)

print(f"Total pairs missed: {len(missed)}")
print(f"Pairs missed where BOTH entities exist in our subset: {len(verifiable_misses)}")

# 2. Look at the text of 3 actual misses to see why BM25 failed
if len(verifiable_misses) > 0:
    print("\n" + "="*40)
    print("INSPECTING 3 MISSED PAIRS")
    print("="*40)
    
    # Create a quick lookup dictionary for names and addresses
    lookup_df = pd.concat([
        s1_normalized[['entity_id', 'business_name', 'business_address']],
        s2_normalized[['entity_id', 'business_name', 'business_address']],
        s3_normalized[['entity_id', 'business_name', 'business_address']]
    ]).set_index('entity_id')
    
    for i, pair in enumerate(verifiable_misses[:3]):
        id_A, id_B = pair
        print(f"\n--- Missed Pair {i+1} ---")
        print(f"Entity A ({id_A}): {lookup_df.loc[id_A, 'business_name']} | {lookup_df.loc[id_A, 'business_address']}")
        print(f"Entity B ({id_B}): {lookup_df.loc[id_B, 'business_name']} | {lookup_df.loc[id_B, 'business_address']}")

In [ ]:


# Load the full ground truth directly
g = pd.read_csv("../student_resource/dataset/train/train_ground_truth.tsv", sep='\t')

print("=== GROUND TRUTH PREVIEW ===")
print(g.head())

# Check the prefixes of the IDs in the ground truth
print("\n=== ID PREFIX DISTRIBUTION ===")
# Assuming the columns are actually 'id1' and 'id2'
if 'source1_entity_id' in g.columns and 'matched_entity_ids' in g.columns:
    print("source1_entity_id Prefixes:")
    print(g['source1_entity_id'].astype(str).str.split('-').str[0].value_counts())
    
    print("\nmatched_entity_ids Prefixes:")
    print(g['matched_entity_ids'].astype(str).str.split('-').str[0].value_counts())
else:
    print(f"Columns are actually: {g.columns.tolist()}")

## Updated Validation code

In [ ]:
import pandas as pd

def validate_blocking_recall(candidates_df, ground_truth_df, s1_df, s2_df, s3_df):
    """
    Calculates fair recall by ensuring we only evaluate pairs where 
    both entities actually exist in the current sample dataframes.
    """
    print("Extracting available IDs from current sample subsets...")
    available_ids = set(s1_df['entity_id']).union(set(s2_df['entity_id'])).union(set(s3_df['entity_id']))
    
    print("Parsing raw ground truth clusters into verifiable pairs...")
    true_pairs = set()
    
    # Drop rows with NaN in matched_entity_ids
    valid_gt = ground_truth_df.dropna(subset=['matched_entity_ids'])
    
    for _, row in valid_gt.iterrows():
        s1_id = str(row['source1_entity_id']).strip()
        
        # Split the comma-separated string into a list of individual IDs
        matches = str(row['matched_entity_ids']).split(',')
        
        for match_id in matches:
            match_id = match_id.strip()
            if match_id:
                # ONLY count the pair if BOTH entities physically exist in our sample
                if s1_id in available_ids and match_id in available_ids:
                    pair = tuple(sorted([s1_id, match_id]))
                    true_pairs.add(pair)
                
    total_true = len(true_pairs)
    print(f"Total Verifiable True Matches to find: {total_true}")
    
    if total_true == 0:
        print("ERROR: No valid ground truth pairs exist in the current subsets.")
        return set(), set()

    print("Standardizing generated candidate pairs...")
    # Fast set comprehension for pairs
    candidate_pairs = set(
        tuple(sorted([str(a), str(b)])) 
        for a, b in zip(candidates_df['entity_A'], candidates_df['entity_B'])
    )
        
    total_candidates = len(candidate_pairs)
    print(f"Total Candidate Pairs Generated: {total_candidates}")
    
    # Calculate Recall
    captured_pairs = true_pairs.intersection(candidate_pairs)
    captured_count = len(captured_pairs)
    
    recall = (captured_count / total_true) * 100
    
    print("\n" + "="*30)
    print("BLOCKING PIPELINE METRICS")
    print("="*30)
    print(f"-> True Matches Captured: {captured_count} / {total_true}")
    print(f"-> RECALL: {recall:.2f}%")
    
    # Calculate Theoretical Search Space Reduction
    max_comps = (len(s1_df) * len(s2_df)) + (len(s1_df) * len(s3_df)) + (len(s2_df) * len(s3_df))
    reduction_ratio = 100 - ((total_candidates / max_comps) * 100) if max_comps > 0 else 0
    print(f"-> Search Space Reduction: ~{reduction_ratio:.4f}% (down from {max_comps:,} pairs)")
    
    return captured_pairs, true_pairs - candidate_pairs

# Execute validation on the new sample data
captured, missed = validate_blocking_recall(
    global_candidates_df, 
    g, 
    s1_normalized, 
    s2_normalized, 
    s3_normalized
)

## Validation after including index 3

In [ ]:
import pandas as pd
from src.blocking import run_multi_index_generation, run_multi_source_geo_blocking

def generate_global_candidate_pool(s1, s2, s3, top_k=15):
    """Executes Index 1 & 2 across all pairs and standardizes columns."""
    print("="*40)
    print("STAGE 1-3: GLOBAL BM25 INDICES")
    print("="*40)
    
    pairs_12 = run_multi_index_generation(s1, s2, top_k)
    pairs_13 = run_multi_index_generation(s1, s3, top_k)
    pairs_23 = run_multi_index_generation(s2, s3, top_k)
    
    std_cols = ['entity_A', 'entity_B', 'bm25_text_score', 'bm25_phonetic_score']
    
    if not pairs_12.empty: pairs_12.columns = std_cols
    if not pairs_13.empty: pairs_13.columns = std_cols
    if not pairs_23.empty: pairs_23.columns = std_cols
    
    master_pool = pd.concat([pairs_12, pairs_13, pairs_23], ignore_index=True)
    return master_pool.drop_duplicates(subset=['entity_A', 'entity_B'])

# 1. Run Global Search (Index 1 & 2)
global_candidates_df = generate_global_candidate_pool(s1_normalized, s2_normalized, s3_normalized, top_k=15)

# 2. Run Local Geo Search (Index 3)
geo_candidates_df = run_multi_source_geo_blocking(s1_normalized, s2_normalized, s3_normalized, top_k=15)

# 3. Merge All Nets Together
print("\nMerging Local Geo matches into Master Candidate Pool...")
final_master_pool = pd.merge(
    global_candidates_df, 
    geo_candidates_df, 
    on=['entity_A', 'entity_B'], 
    how='outer'
).fillna(0.0)

print(f"Final Candidate Pool Size: {len(final_master_pool)}")

# 4. Final Validation check 
captured, missed = validate_blocking_recall(
    final_master_pool, 
    g, 
    s1_normalized, 
    s2_normalized, 
    s3_normalized
)

In [ ]:
import pandas as pd

def generate_baseline_submission(master_pool_df, s1_df, output_path="matching_results.tsv"):
    print("Generating baseline submission for the leaderboard...")
    
    # 1. Naive Thresholding (Surrogate for our future LightGBM model)
    # We only keep pairs with a reasonably strong signal to avoid tanking our precision.
    strong_matches = master_pool_df[
        (master_pool_df['bm25_text_score'] > 15.0) | 
        (master_pool_df['geo_ngram_score'] > 0.5)
    ].copy()
    
    # 2. Group by Source 1 ID and create the comma-separated list of S2/S3 matches
    grouped = strong_matches.groupby('entity_A')['entity_B'].apply(
        lambda x: ','.join(x.dropna().unique())
    ).reset_index()
    
    grouped = grouped.rename(columns={
        'entity_A': 'source1_entity_id', 
        'entity_B': 'matched_entity_ids'
    })
    
    # 3. Ensure ALL Source 1 entities are present in the final file
    # As required: "One row per Source 1 entity, empty list means no match"
    submission_df = pd.DataFrame({'source1_entity_id': s1_df['entity_id']})
    submission_df = pd.merge(submission_df, grouped, on='source1_entity_id', how='left')
    
    # 4. Fill missing matches with an empty string
    submission_df['matched_entity_ids'] = submission_df['matched_entity_ids'].fillna("")
    
    # 5. Export as TSV
    submission_df.to_csv(output_path, sep='\t', index=False)
    
    print(f"Saved submission to: {output_path}")
    print(f"Total rows in submission: {len(submission_df)} (Should match S1 row count)")
    print(f"S1 entities with at least one match: {len(submission_df[submission_df['matched_entity_ids'] != ''])}")
    
    return submission_df

# Generate the file using your full normalized S1 dataframe and the final candidate pool
# Note: You should run this on the full 'train' datasets, not just the 10k samples, 
# to get a real leaderboard score.
submission = generate_baseline_submission(
    final_master_pool, 
    s1_normalized, 
    output_path="matching_results.tsv"
)

## Removing Redundancy

In [6]:
from src.blocking import generate_global_candidate_pool, run_multi_source_geo_blocking

# 1. Global Scan (Shared tokenization & single-build indices)
# On SageMaker ml.r5.4xlarge: chunk_size=100000, n_threads=16
# Locally on M4: chunk_size=50000, n_threads=8
global_candidates_df = generate_global_candidate_pool(
    s1_normalized, 
    s2_normalized, 
    s3_normalized, 
    top_k=15, 
    chunk_size=100000, 
    n_threads=16
)

# 2. Local Spatial Safety Net
geo_candidates_df = run_multi_source_geo_blocking(
    s1_normalized, 
    s2_normalized, 
    s3_normalized, 
    top_k=15
)

# 3. Outer Merge into the Master Candidate Pool
print("\nMerging Local Geo matches into Master Candidate Pool...")
final_master_pool = pd.merge(
    global_candidates_df, 
    geo_candidates_df, 
    on=['entity_A', 'entity_B'], 
    how='outer'
).fillna(0.0)

print(f"Final Candidate Pool Size: {len(final_master_pool)}")

STAGES 1-3: GLOBAL BM25 CANDIDATE EXTRACTION

[INDIA | TEXT] S1: 24093 | S2: 27356 | S3: 27749
  -> Building Target S2 Index...


Split strings:   0%|          | 0/27356 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/27356 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/27356 [00:00<?, ?it/s]

  -> Building Target S3 Index...


...

  -> Querying S1 across target indices in chunks of 100000...


Split strings:   0%|          | 0/35907 [00:00<?, ?it/s]

     ...Processed 35907/35907
  -> Querying S2 against existing S3 Index in chunks of 100000...


Split strings:   0%|          | 0/40348 [00:00<?, ?it/s]

     ...Processed 40348/40348
  -> Generated 1682430 candidate pairs in 14.63s.
  -> Merging text & phonetic candidates for US...

Total Unified Global BM25 Pairs: 5278277

STAGE 4: INDEX 3 (LOCAL GEO BLOCKING)

[S1 vs S2]
  -> Sweeping 61 shared geographic blocks...

[S1 vs S3]
  -> Sweeping 30 shared geographic blocks...

[S2 vs S3]
  -> Sweeping 49 shared geographic blocks...

Total Local Geo Pairs Generated: 2042805

Merging Local Geo matches into Master Candidate Pool...
Final Candidate Pool Size: 6743628


In [7]:
import pandas as pd

def validate_blocking_recall(candidates_df, ground_truth_df, s1_df, s2_df, s3_df):
    """
    Calculates fair recall by ensuring we only evaluate pairs where 
    both entities actually exist in the current sample dataframes.
    """
    print("Extracting available IDs from current sample subsets...")
    available_ids = set(s1_df['entity_id']).union(set(s2_df['entity_id'])).union(set(s3_df['entity_id']))
    
    print("Parsing raw ground truth clusters into verifiable pairs...")
    true_pairs = set()
    
    # Drop rows with NaN in matched_entity_ids
    valid_gt = ground_truth_df.dropna(subset=['matched_entity_ids'])
    
    for _, row in valid_gt.iterrows():
        s1_id = str(row['source1_entity_id']).strip()
        
        # Split the comma-separated string into a list of individual IDs
        matches = str(row['matched_entity_ids']).split(',')
        
        for match_id in matches:
            match_id = match_id.strip()
            if match_id:
                # ONLY count the pair if BOTH entities physically exist in our sample
                if s1_id in available_ids and match_id in available_ids:
                    pair = tuple(sorted([s1_id, match_id]))
                    true_pairs.add(pair)
                
    total_true = len(true_pairs)
    print(f"Total Verifiable True Matches to find: {total_true}")
    
    if total_true == 0:
        print("ERROR: No valid ground truth pairs exist in the current subsets.")
        return set(), set()

    print("Standardizing generated candidate pairs...")
    # Fast set comprehension for pairs
    candidate_pairs = set(
        tuple(sorted([str(a), str(b)])) 
        for a, b in zip(candidates_df['entity_A'], candidates_df['entity_B'])
    )
        
    total_candidates = len(candidate_pairs)
    print(f"Total Candidate Pairs Generated: {total_candidates}")
    
    # Calculate Recall
    captured_pairs = true_pairs.intersection(candidate_pairs)
    captured_count = len(captured_pairs)
    
    recall = (captured_count / total_true) * 100
    
    print("\n" + "="*30)
    print("BLOCKING PIPELINE METRICS")
    print("="*30)
    print(f"-> True Matches Captured: {captured_count} / {total_true}")
    print(f"-> RECALL: {recall:.2f}%")
    
    # Calculate Theoretical Search Space Reduction
    max_comps = (len(s1_df) * len(s2_df)) + (len(s1_df) * len(s3_df)) + (len(s2_df) * len(s3_df))
    reduction_ratio = 100 - ((total_candidates / max_comps) * 100) if max_comps > 0 else 0
    print(f"-> Search Space Reduction: ~{reduction_ratio:.4f}% (down from {max_comps:,} pairs)")
    
    return captured_pairs, true_pairs - candidate_pairs

# Execute validation on the deduplicated master pool
captured, missed = validate_blocking_recall(
    final_master_pool, 
    gt, 
    s1_normalized, 
    s2_normalized, 
    s3_normalized
)

Extracting available IDs from current sample subsets...
Parsing raw ground truth clusters into verifiable pairs...
Total Verifiable True Matches to find: 36703
Standardizing generated candidate pairs...
Total Candidate Pairs Generated: 6743628

BLOCKING PIPELINE METRICS
-> True Matches Captured: 36463 / 36703
-> RECALL: 99.35%
-> Search Space Reduction: ~99.9476% (down from 12,873,688,296 pairs)


In [8]:
import pandas as pd

def generate_baseline_submission(master_pool_df: pd.DataFrame, s1_df: pd.DataFrame, output_path: str = "matching_results.tsv"):
    """
    Generates a leaderboard-ready TSV file from the candidate pool using naive thresholds.
    Ensures exact schema compliance: exactly one row per S1 entity, no duplicates.
    """
    print("Generating baseline submission for the leaderboard...")
    
    # 1. Naive Thresholding (Surrogate for LightGBM)
    # We now explicitly include the phonetic score threshold alongside text and geo scores.
    strong_matches = master_pool_df[
        (master_pool_df['bm25_text_score'] > 15.0) | 
        (master_pool_df['bm25_phonetic_score'] > 10.0) | 
        (master_pool_df.get('geo_ngram_score', pd.Series(0, index=master_pool_df.index)) > 0.5)
    ].copy()
    
    # 2. Group by Source 1 ID and create the comma-separated list of unique S2/S3 matches
    grouped = strong_matches.groupby('entity_A')['entity_B'].apply(
        lambda x: ','.join(x.dropna().unique())
    ).reset_index()
    
    grouped = grouped.rename(columns={
        'entity_A': 'source1_entity_id', 
        'entity_B': 'matched_entity_ids'
    })
    
    # 3. Ensure ALL Source 1 entities are present in the final file
    # This enforces the rule that every S1 entity must have exactly one row.
    submission_df = pd.DataFrame({'source1_entity_id': s1_df['entity_id']})
    submission_df = pd.merge(submission_df, grouped, on='source1_entity_id', how='left')
    
    # 4. Fill missing matches with an empty string
    # Entities with no matches are explicitly left blank.
    submission_df['matched_entity_ids'] = submission_df['matched_entity_ids'].fillna("")
    
    # 5. Export as TSV
    submission_df.to_csv(output_path, sep='\t', index=False)
    
    print(f"Saved submission to: {output_path}")
    print(f"Total rows in submission: {len(submission_df)} (Should match S1 row count: {len(s1_df)})")
    print(f"S1 entities with at least one match: {len(submission_df[submission_df['matched_entity_ids'] != ''])}")
    
    return submission_df

# Generate the final matching_results.tsv
submission = generate_baseline_submission(
    final_master_pool, 
    s1_normalized, 
    output_path="matching_results.tsv"
)

Generating baseline submission for the leaderboard...
Saved submission to: matching_results.tsv
Total rows in submission: 60000 (Should match S1 row count: 60000)
S1 entities with at least one match: 15089
